# HAWC2 I/O sensor search demo

This notebook demonstrates the current `ReadHawc2` interface for loading HAWC2 HDF5 (GTSDF) results and selecting sensors.

- Calling `hawc2io(...)` returns the selected samples as a NumPy array.
- Calling `hawc2io.ViewSensors(...)` returns the same samples as a pandas DataFrame whose columns contain sensor metadata.
- Searches are case-insensitive and ignore spaces. Multiple filters are combined.
- HTC searches are available only when the HDF5 file contains `htc_input` metadata.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from wetb.hawc2.Hawc2io import ReadHawc2

from wetb.hawc2.Hawc2Output import Hawc2Output

ModuleNotFoundError: No module named 'wetb.hawc2.Hawc2Output'

## Open the bundled result file

The path expression works both when Jupyter starts in this directory and when it starts at the repository root.

In [6]:
filename = Path("IEA15_htc_input_test.hdf5")
if not filename.exists():
    filename = Path(r"c:\Users\ibesi\repos\WindEnergyToolbox\wetb\hawc2\hawc2io_demo\IEA15_htc_input_test.hdf5")

hawc2io = Hawc2Output(str(filename))
print(f"Samples: {hawc2io.NrSc}, channels: {hawc2io.NrCh}")

NameError: name 'Hawc2Output' is not defined

## Inspect sensor metadata

`sensor_search()` returns metadata only; it does not load time-series data. With no filters it lists every channel.

In [5]:
sensor_metadata = hawc2io.sensor_search()
sensor_metadata.head(10)

AttributeError: 'ReadHawc2' object has no attribute 'sensor_search'

## Load raw arrays

Pass channel IDs positionally or search using `name`, `unit`, `desc`, `label`, or (when available) `htc`. The result is always a two-dimensional NumPy array, including single-channel selections.

In [ ]:
by_channel_id = hawc2io([0, 12])
tower_force = hawc2io(name="test force vector tower")
tower_base_moments = hawc2io(desc=["MomentMx Mbdy:tower", "MomentMy Mbdy:tower", "MomentMz Mbdy:tower"])

print(type(tower_force), tower_force.shape)
print(tower_base_moments.shape)

## Load a metadata-rich DataFrame

Use `ViewSensors` when column labels, units, and descriptions are needed. Its columns are a pandas MultiIndex built from the matching sensor metadata.

In [ ]:
tower_force_df = hawc2io.ViewSensors(name="test force vector tower")
display(tower_force_df.head())
tower_force_df.columns.to_frame(index=False)

The raw array and DataFrame contain the same selected samples.

In [ ]:
np.testing.assert_allclose(tower_force, tower_force_df.to_numpy())

## Plot selected channels

In [ ]:
moment_df = hawc2io.ViewSensors(
    desc=["MomentMx Mbdy:tower", "MomentMy Mbdy:tower", "MomentMz Mbdy:tower"]
)

ax = moment_df.plot(figsize=(10, 4))
ax.set_title("Tower-base moments")
ax.set_xlabel("Sample")
ax.set_ylabel("Moment")
plt.tight_layout()

## HTC metadata

The bundled file does not contain HTC input metadata. For an updated HDF5 file that does, `hawc2io.sensor_search.df` includes an `HTC_input` column and the same query can be passed to either `hawc2io(...)` or `hawc2io.ViewSensors(...)`.

In [ ]:
if "HTC_input" in hawc2io.sensor_search.df.columns:
    htc_matches = hawc2io.ViewSensors(htc="aero omega")
    display(htc_matches.head())
else:
    print("This file has no HTC_input metadata; HTC search is unavailable.")

## Empty searches

A search with no matches returns an empty selection rather than raising an exception.

In [ ]:
empty = hawc2io(name="non-existing sensor")
print(empty.shape)
assert empty.shape == (hawc2io.NrSc, 0)